# Orbit Wars — публикация данных как Kaggle Dataset

Создаёт (или обновляет новой версией) Kaggle Dataset с локальным `data.zip`
(в нём `data/sft.full_send.jsonl` + промежуточные `samples.*`). Этот датасет
потом подключается во **втором** ноутбуке (`kaggle_02_train_eval`) и
распаковывается в `./data`.

**Где запускать:** локально, там где лежит `data.zip` (≈570 МБ) — внутри
Kaggle-кернела локального файла нет. Нужен токен Kaggle API
(`kaggle.com → Settings → API → Create New Token` → `~/.kaggle/kaggle.json`,
`chmod 600`), либо переменные окружения `KAGGLE_USERNAME` / `KAGGLE_KEY`.


## 1. Параметры и поиск `data.zip`

In [ ]:
from pathlib import Path

# Слаг датасета: <username>/<name>. username подставится из kaggle.json/env ниже.
KAGGLE_USERNAME = None              # None -> возьмём из ~/.kaggle/kaggle.json или env
DATASET_NAME    = "orbit-wars-sft-data"
DATASET_TITLE   = "Orbit Wars SFT data"

def _find_up(name, start=None):
    p = (start or Path.cwd()).resolve()
    for d in [p, *p.parents]:
        if (d / name).exists():
            return d / name
    raise FileNotFoundError(f"{name} не найден вверх от {p}")

DATA_ZIP  = _find_up("data.zip")
REPO_ROOT = DATA_ZIP.parent
print("data.zip :", DATA_ZIP, f"({DATA_ZIP.stat().st_size / 1e6:.0f} MB)")
print("repo root:", REPO_ROOT)


## 2. Kaggle CLI + учётные данные

In [ ]:
import os, sys, json, subprocess

# kaggle CLI (в .venv проекта уже есть; иначе ставим)
try:
    import kaggle  # noqa: F401
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "kaggle==2.2.1"], check=True)

cred = Path.home() / ".kaggle" / "kaggle.json"
if cred.exists():
    KAGGLE_USERNAME = KAGGLE_USERNAME or json.loads(cred.read_text())["username"]
elif os.environ.get("KAGGLE_USERNAME"):
    KAGGLE_USERNAME = KAGGLE_USERNAME or os.environ["KAGGLE_USERNAME"]
else:
    raise SystemExit(
        "Нет kaggle-кред: положи ~/.kaggle/kaggle.json (chmod 600) или задай "
        "env KAGGLE_USERNAME/KAGGLE_KEY. Токен: kaggle.com -> Settings -> API.")

DATASET_SLUG = f"{KAGGLE_USERNAME}/{DATASET_NAME}"
print("датасет:", DATASET_SLUG)


## 3. Staging-папка

Kaggle грузит содержимое папки. Кладём в неё `dataset-metadata.json` и сам
`data.zip` (симлинком, без копирования 570 МБ; если ФС не умеет симлинки —
копией).

In [ ]:
import shutil

stage = REPO_ROOT / "kaggle_dataset_upload"
stage.mkdir(exist_ok=True)

(stage / "dataset-metadata.json").write_text(json.dumps({
    "title": DATASET_TITLE,         # <= 50 символов
    "id": DATASET_SLUG,             # обязателен: username/name
    "licenses": [{"name": "CC0-1.0"}],
}, ensure_ascii=False, indent=2))

link = stage / "data.zip"
if link.exists() or link.is_symlink():
    link.unlink()
try:
    link.symlink_to(DATA_ZIP)
except OSError:
    shutil.copy2(DATA_ZIP, link)

print("staging:", stage)
print(sorted(p.name for p in stage.iterdir()))


## 4. Создать или обновить датасет

Если датасет уже существует — заливаем новую **версию**, иначе **создаём**.
Загрузка ~570 МБ занимает несколько минут.

In [ ]:
def _run(cmd):
    print("+", " ".join(cmd))
    r = subprocess.run(cmd, capture_output=True, text=True)
    print(r.stdout)
    if r.stderr:
        print(r.stderr)
    return r

ls = subprocess.run(["kaggle", "datasets", "list", "--user", KAGGLE_USERNAME,
                     "-s", DATASET_NAME], capture_output=True, text=True)
exists = DATASET_SLUG in ls.stdout

if exists:
    print(f"[update] {DATASET_SLUG} уже есть -> новая версия")
    r = _run(["kaggle", "datasets", "version", "-p", str(stage),
              "-m", "update data.zip", "--dir-mode", "skip"])
else:
    print(f"[create] создаём {DATASET_SLUG}")
    r = _run(["kaggle", "datasets", "create", "-p", str(stage), "--dir-mode", "skip"])

if r.returncode == 0:
    print(f"\nГотово: https://www.kaggle.com/datasets/{DATASET_SLUG}")
else:
    print("\nОшибка — см. вывод выше.")


## Дальше

Во втором ноутбуке (`kaggle_02_train_eval`) нажми **Add Input** и подключи
этот датасет (`<username>/orbit-wars-sft-data`). Он смонтируется как
`/kaggle/input/orbit-wars-sft-data/data.zip` — ноутбук сам распакует его в
`./data`.
